# hCKSAAP_UbSite human: residual hybrid benchmark on Kaggle

This notebook trains the Short-Range Expert and Long-Context Expert on the published hCKSAAP_UbSite human training set, fits the residual fusion from protein-grouped out-of-fold predictions, refits both experts, and compares with the authors' dataset-specific checkpoint on matched published test sites.

Select **GPU T4 x2** and enable **Internet** in Kaggle settings. Add a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository; `HF_TOKEN` is optional. Training uses five outer folds and 15 epochs per candidate without early stopping. Separate fold workers use the two T4s concurrently; the final refit uses one T4. The released dataset files come from the Git submodule, and the public checkpoint is downloaded and checksum-verified below.

Enable Kaggle notebook **Persistence → Files only** before training. Intermediate files are written to `/kaggle/working`; for a different account or notebook, attach a saved output dataset and set `RESUME_RUN_DIR` in the paths cell. Kaggle persistence is best effort, so save a version or export the run directory at milestones.

## Repository and credentials

In [ ]:
from pathlib import Path
import os, subprocess, sys

from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
HF_TOKEN = secret('HF_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add the read-only GITHUB_TOKEN secret to this Kaggle notebook.')

REPO_ROOT = Path('/kaggle/working/Ubiquitination-Sites-Prediction')
REPO_REF = None  # Set to the printed commit SHA when restoring a prior run.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN'] = GITHUB_TOKEN
askpass = Path('/kaggle/working/ubipred_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
askpass.chmod(0o700)
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules',
                        'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
                        str(REPO_ROOT)], check=True, env=git_env)
    elif REPO_REF is None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
    if REPO_REF is not None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', REPO_REF],
                       check=True, env=git_env)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
                   check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/kaggle/working/ubipred_huggingface_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

## Dependencies and accelerator check

In [ ]:
%pip install -q -r experiment/requirements-colab.txt h5py gdown
%pip uninstall -y torchao

In [ ]:
import json, torch

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 before training.')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run_live(command, env=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Paths, source preflight and released checkpoint

The released source files are already in the Git submodule. The checkpoint is taken from an attached Kaggle dataset if present, or downloaded from the authors' public Drive folder. Its SHA-256 is checked before use. The independent test records are not loaded during this audit.

In [ ]:
import hashlib, shutil

CONFIG = Path('experiment/configs/benchmarks/hcksaap_residual_v1.json')
BENCHMARK = 'hcksaap'
RUN_DIR = Path('/kaggle/working/mmubipred_benchmark_runs') / BENCHMARK / 'run'
SOURCE_DIR = Path('/kaggle/working/mmubipred_benchmark_sources') / BENCHMARK
MODEL_FILE = SOURCE_DIR / 'DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5'
MODEL_DRIVE_ID = '1v6u6N6JpqJs8bi_3wLhp5gcbS7uxsAwD'
MODEL_SHA256 = 'fef6859092626f94e6a7a70e644c37f2f68bf93ec00f251c2003458dccb90526'

# To restore a previously saved Kaggle output, attach it as Input and set this path
# to the directory ending in /hcksaap/run. Leave None for a new or persisted session.
RESUME_RUN_DIR = None
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not (previous / 'protocol.json').is_file():
        raise FileNotFoundError(f'Prior run protocol missing: {previous}')
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise RuntimeError(f'Current run directory is not empty: {RUN_DIR}')
    shutil.copytree(previous, RUN_DIR, dirs_exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

if not MODEL_FILE.is_file():
    attached = list(Path('/kaggle/input').rglob(MODEL_FILE.name))
    if len(attached) > 1:
        raise RuntimeError(f'Multiple attached copies of the baseline: {attached}')
    if attached:
        shutil.copy2(attached[0], MODEL_FILE)
    else:
        import gdown
        gdown.download(id=MODEL_DRIVE_ID, output=str(MODEL_FILE), quiet=False)
if not MODEL_FILE.is_file() or sha256(MODEL_FILE) != MODEL_SHA256:
    raise RuntimeError(f'Baseline checkpoint missing or checksum mismatch: {MODEL_FILE}')
print('Verified checkpoint:', MODEL_FILE)
print('Run directory:', RUN_DIR)

def stage_command(name, *extra):
    return [sys.executable, '-u', 'experiment/scripts/run_dataset_benchmark.py', name,
            '--config', CONFIG, '--run-dir', RUN_DIR, '--source-dir', SOURCE_DIR,
            '--model-file', MODEL_FILE, *extra]

def stage(name, *extra):
    run_live(stage_command(name, *extra))

stage('audit')
audit = json.loads((RUN_DIR / 'source_audit.json').read_text())
display(audit)
assert audit['source_preflight_passed'], audit['blockers']

## Prepare training context

The source adapter validates the published training windows against UniProt protein sequences and requires at least 95% usable context. It does not access the published test split.

In [ ]:
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', 'experiment/tests',
          '-p', 'test_dataset_benchmark.py'])
stage('prepare-train')
display(json.loads((RUN_DIR / 'train_preparation.json').read_text()))

## Five-fold development on two T4 GPUs

GPU 0 processes folds 0, 2 and 4; GPU 1 processes folds 1 and 3. Each worker writes to separate fold directories. Rerunning this cell reuses completed fold predictions and resumes incomplete candidates from the last completed epoch. When both workers finish, a final pass assembles all out-of-fold predictions and fits the residual fusion.

In [ ]:
import threading, time

def run_parallel_development():
    assignments = [(0, [0, 2, 4]), (1, [1, 3])]
    processes, readers = [], []
    def stream_output(process, gpu):
        for line in process.stdout:
            print(f'[T4 {gpu}] {line}', end='', flush=True)
    try:
        for gpu, folds in assignments:
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
            env['OMP_NUM_THREADS'] = '2'
            env['TOKENIZERS_PARALLELISM'] = 'false'
            command = stage_command('develop', *[part for fold in folds
                                                  for part in ('--fold', str(fold))])
            print(f'Starting GPU {gpu} on folds {folds}', flush=True)
            process = subprocess.Popen([str(part) for part in command], env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1)
            processes.append((gpu, process))
            reader = threading.Thread(target=stream_output, args=(process, gpu), daemon=True)
            reader.start()
            readers.append(reader)
        while True:
            failed = [(gpu, p.returncode) for gpu, p in processes
                      if p.poll() is not None and p.returncode != 0]
            if failed:
                raise RuntimeError(f'Fold worker failed: {failed}')
            if all(p.poll() == 0 for _, p in processes):
                break
            time.sleep(1)
    except BaseException:
        for _, process in processes:
            if process.poll() is None:
                process.terminate()
        for _, process in processes:
            process.wait()
        raise
    finally:
        for reader in readers:
            reader.join(timeout=5)

run_parallel_development()
stage('develop')
display(json.loads((RUN_DIR / 'development_summary.json').read_text()))

## Full training-set refit and model lock

Each expert is refitted once on the eligible published training data for the median best epoch selected in development. The fusion weights and decision threshold are then frozen.

In [ ]:
stage('refit')
display(json.loads((RUN_DIR / 'model_lock.json').read_text()))

## Published test preparation

Run only after the model lock exists. Both the released checkpoint and the newly trained hybrid will be evaluated on the same retained, context-valid test sites.

In [ ]:
assert (RUN_DIR / 'model_lock.json').exists(), 'Complete the refit first.'
stage('prepare-test', '--allow-test')
display(json.loads((RUN_DIR / 'test_preparation.json').read_text()))

## Legacy Keras comparator

The published HDF5 checkpoint uses legacy Keras layers. Install the matching `tf-keras` package without replacing Kaggle's existing TensorFlow dependencies.

In [ ]:
import importlib.metadata as metadata

try:
    tf_version = metadata.version('tensorflow')
except metadata.PackageNotFoundError as error:
    raise RuntimeError('TensorFlow is absent from this Kaggle image; install a compatible TensorFlow CPU package before evaluation.') from error
legacy_requirement = 'tf-keras==' + '.'.join(tf_version.split('.')[:2]) + '.*'
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', legacy_requirement])

## Matched test comparison

The evaluation uses the locked hybrid, the authors' dataset-specific checkpoint, fixed threshold 0.5 and protein-cluster paired bootstrap. Keep the output together with the run directory.

In [ ]:
stage('evaluate', '--allow-test')
result = json.loads((RUN_DIR / 'test_results.json').read_text())
display(result['metrics'])
display(result['paired_hybrid_vs_paper'])

## Resumption and output transfer

With **Persistence → Files only**, rerun setup, paths and the interrupted stage after a session restart. Completed fold predictions are skipped; an interrupted epoch is repeated. For a different Kaggle account, save or export the entire `/kaggle/working/mmubipred_benchmark_runs/hcksaap/run` directory, attach it to the new notebook as Input, and set `RESUME_RUN_DIR` in the paths cell. Use the same repository commit printed during the original run; changing the training implementation invalidates the recorded protocol. The public checkpoint can be downloaded again.

Kaggle's `/kaggle/working` files are not guaranteed to survive every failure. Save a notebook version with outputs or download a run snapshot before the session ends. The two-T4 parallelism applies to development folds; refit and final inference are single-process stages.